# **Importing Libraries**

In [ ]:
# importing necessary libraries / dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import scipy.stats as stats
from scipy.stats import f_oneway
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.pipeline import Pipeline
from sklearn.manifold import TSNE
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import label_binarize
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import quantile_transform
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import shapiro, normaltest, skew, kurtosis
pd.set_option('display.max_columns', None)

# **Loading the DataFrame**

In [42]:
# Retrieving dataset

df = pd.read_csv('/Users/omaralhajjaj/Projects/HCV proj/COM725/HCV-Egy-Data.csv')

# **Exploratory Data Analysis**

In [ ]:
# first 5 rows of the dataset

df.head()

In [ ]:
# viewing column names to understand DataFrame contents and finding null values

columns = df.columns
print(columns)

df_info = pd.DataFrame(df.info()).T
print(df_info)

In [ ]:
#	loop to gather column names in a list to collect numerical columns

numerical_columns = []
for column in df.columns:
  # Using try-except block to handle cases where max() might not be applicable
  try:
    if df[column].max() > 2:
      numerical_columns.append(column)
  except TypeError:
    continue

# dropping Staging column as its an encoded categorical column with 4 categories
# and dropping Grading as the column requires surgical intervention

columns_to_remove = ['Baseline histological Grading', 'Baselinehistological staging']
numerical_columns = [col for col in numerical_columns if col not in columns_to_remove]
print(numerical_columns)

df_numeric = df[numerical_columns]
print('\n')
df_numeric.describe()

In [ ]:
describe = df_numeric.describe()
describe.to_excel('my_description.xlsx', index=False)

In [ ]:
# value counts to find distribution of categorical variables
categorical_columns = []

for column in df.columns:
  if column not in numerical_columns:
    categorical_columns.append(column)
columns_to_remove2 = ['Baseline histological Grading', 'Baselinehistological staging']
categorical_columns = [col for col in categorical_columns if col not in columns_to_remove2]

print(categorical_columns)

df_categorical = df[categorical_columns]

# Visualizing correlations and relationships

In [ ]:
# visualising linear relationships in the dataset

correlations = df.corr()
plt.figure(figsize=(25, 12))
mask_1 = np.triu(correlations)
sns.heatmap(correlations, annot=True, mask = mask_1, cmap='coolwarm')


The plot here shows that variables have very weak relationships to each other, reflecting a degree of randomness in the data

# **Removing outliers for analysis**

In [ ]:
# Calculate first quartile (25th percentile) for each numerical column
Q1 = df[numerical_columns].quantile(0.25)

# Calculate third quartile (75th percentile)
Q3 = df[numerical_columns].quantile(0.75)

# Compute Interquartile Range (IQR)
IQR = Q3 - Q1

# Identify outliers using IQR rule
outliers = (df[numerical_columns] < (Q1 - 1.5 * IQR)) \
            | (df[numerical_columns] > (Q3 + 1.5 * IQR))

# Remove rows where any column contains an outlier
df_clean = df[~outliers.any(axis=1)]

# View cleaned dataset
df_clean

In [ ]:
# Identify rows that contain at least one outlier across numerical columns
outlier_row_mask = outliers.any(axis=1)

# Keep only the outlier values (non-outliers become NaN)
outlier_values = df[numerical_columns].where(outliers)

# Filter to rows that actually contain outliers
outlier_values = outlier_values[outlier_row_mask]

# Outlier removal using isolation forrest

In [ ]:
# Initialize Isolation Forest for anomaly detection

iso_outliers = IsolationForest(n_estimators=100, max_samples=256, contamination=0.05,
                               random_state = 42)

# Fit model on numerical features and predict anomalies

iso_pred = iso_outliers.fit_predict(df[numerical_columns])

# Create binary outlier flag column (True = outlier)
df['outlier'] = (iso_pred == -1)

In [ ]:
# Creating a new dataset free from outliers

df_clean_iso = df[~df['outlier']]
df_clean_iso = df_clean_iso.drop(index=outlier_values.index, errors='ignore')
df_clean_iso.drop(columns = 'outlier')

In [ ]:
# testing correlations after outlier removal

clean_corr = df_clean_iso.corr()

mask = np.triu(np.ones_like(clean_corr))

plt.figure(figsize=(25, 12))

sns.heatmap(clean_corr, annot=True, cmap='coolwarm', mask=mask)

plt.show()

Outlier removal has reflected no effect on the relationships between variables in the dataset

# **Normality and data distribution analysis**

In [ ]:
# normality tests for numerical columns

normality_results = []

for col in numerical_columns:
    data = df_clean_iso[col].dropna()

    if len(data) < 3:
        continue

    stat_sw, p_sw = shapiro(data)

    try:
        stat_dp, p_dp = normaltest(data)
    except:
        stat_dp, p_dp = np.nan, np.nan

    normality_results.append({
        "Column": col,
        "Mean": data.mean(),
        "Median": data.median(),
        "Std": data.std(),
        "Skewness": skew(data),
        "Kurtosis": kurtosis(data),
        "Shapiro_p": p_sw,
        "Shapiro_Normal": p_sw > 0.05,
        "D_Agostino_p": p_dp,
        "D_Agostino_Normal": p_dp > 0.05 if not np.isnan(p_dp) else np.nan
    })

normality_df = pd.DataFrame(normality_results)
normality_df

Results show that none of the numerical variables in the dataset follow a normal-bell curve distribution

# Q-Q plot

In [ ]:
# Q-Q plots for numerical columns

n_cols = 4
n_rows = math.ceil(len(numerical_columns) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numerical_columns):
    data = df_clean_iso[col].dropna()

    stats.probplot(data, dist="norm", plot=axes[i])
    axes[i].set_title(f"Q-Q Plot: {col}")

# remove empty plots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

QQ-plot confirms abnormal distribution of variables

# Histograms to visualise data distribution

In [ ]:
# distribution plots (4 per row)

n = len(numerical_columns)
fig, axes = plt.subplots((n + 3)//4, 4, figsize=(20, 4*((n + 3)//4)))
axes = axes.flatten()

for i, col in enumerate(numerical_columns):
    sns.histplot(df_clean_iso[col].dropna(), bins='fd', kde=True, ax=axes[i])
    axes[i].set_title(col)

# remove empty subplot if odd number
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

The distribution analysis shows that the majority of numerical variables do not follow a clear normal distribution. Demographic and routine laboratory variables such as age, BMI, WBC, RBC, platelets, AST and ALT values are widely distributed, while HGB appears discretised, suggesting grouped or rounded measurements. The RNA follow-up variables, particularly RNA 12, RNA EOT and RNA EF, show strong right skewness, with many observations concentrated at very low values and fewer patients showing high viral-load values. This indicates that the dataset contains non-normal and skewed clinical measurements, meaning that non-parametric tests and robust machine learning models are more appropriate than methods that assume normally distributed variables.

# **Statistical testing**

In [ ]:
# defining an empty list
alt_columns = []

# collecting ALT columns for statistical testing
for column in df_clean_iso.columns:
  if 'ALT' in column:
    alt_columns.append(column)

# defining an empty list
rna_columns = []

# collecting RNA columns for statistical testing
for column in df_clean_iso.columns:
  if 'RNA' in column:
    rna_columns.append(column)

print(alt_columns)
print('')
print(rna_columns)

# Kruskal wallis test for significance

In [ ]:
# merging test columns
test_columns = alt_columns + rna_columns

# defining empty lidst to gather results
kruskal_results = []

# loop for kruskal wallis test of significance
for col in test_columns:

  test_data = df_clean_iso[[col, 'Baselinehistological staging']]
  stages = test_data['Baselinehistological staging'].unique()
  test_groups = [test_data[test_data['Baselinehistological staging'] == stage][col]
        for stage in sorted(test_data['Baselinehistological staging'].unique())]

  h_stat, p_val = stats.kruskal(*test_groups)

  # calculate effect size
  n = len(test_data)
  k = test_data['Baselinehistological staging'].nunique()
  eta_squared = (h_stat - k + 1) / (n - k)

  kruskal_results.append({
        'Variable': col,
        'H_statistic': h_stat,
        'p_value': p_val,
        'eta_squared': eta_squared})

kruskal_df = pd.DataFrame(kruskal_results)
kruskal_df = kruskal_df.sort_values(by='eta_squared', ascending=False)

print(kruskal_df)

The Kruskal-Wallis results show that none of the ALT or RNA variables differ significantly across fibrosis stages, as all p-values are above 0.05 and the eta-squared values are close to zero, indicating extremely weak explanatory power.



In [ ]:
# transforming data into a normal distribution
qt = QuantileTransformer(output_distribution='normal', random_state=42)

df_transformed = df_clean_iso.copy()
df_transformed[numerical_columns] = qt.fit_transform(df_clean_iso[numerical_columns])

# empty list for results
anova_results = []

test_columns = alt_columns + rna_columns

for col in test_columns:

    data = df_transformed[[col, 'Baselinehistological staging']].dropna()

    groups = [
        data[data['Baselinehistological staging'] == stage][col]
        for stage in sorted(data['Baselinehistological staging'].unique())
    ]

    f_stat, p_value = f_oneway(*groups)

    anova_results.append({
        'Variable': col,
        'F_statistic': f_stat,
        'p_value': p_value
    })

anova_df = pd.DataFrame(anova_results).sort_values(by='p_value')
print(anova_df)

Conclusion:
After applying QuantileTransformer to make the variables closer to a normal distribution, one-way ANOVA still found no significant differences between fibrosis stages for any ALT or RNA variable. This suggests that the weak relationship between these variables and fibrosis stage is not simply caused by non-normality.

In [ ]:
# correlation for transformed data
corr_cols = numerical_columns + ['Baselinehistological staging']
corr_matrix = df_transformed[corr_cols].corr(method='pearson')

# create mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(12,6))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5
)
plt.title("Correlation Matrix (After Quantile Transformation)")
plt.show()

After quantile transformation, the correlation matrix still shows almost no meaningful relationship between the clinical variables and baseline histological staging, although RNA 12, RNA EOT and RNA EF increased in correlation with each other.

# **Machine learning exploration**

In [ ]:
# testing predictive power of different timepoints

early = ['ALT 1', 'ALT4', 'RNA Base', 'RNA 4']
mid   = ['ALT 12', 'RNA 12', 'ALT 24']
late  = ['ALT 36', 'ALT 48', 'RNA EOT']

In [ ]:
def evaluate_model(features):
    X = df_clean_iso[features]
    y = df_clean_iso['Baselinehistological staging']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

In [ ]:
print("Early:", evaluate_model(early))
print("Mid:", evaluate_model(mid))
print("All:", evaluate_model(early + mid + late))

middle time cut-offs have strongest predictive power.

# **Machine learning training and modelling**

# Data preparation

In [ ]:
df_clean_iso = df_clean_iso.drop(columns=['ALT 36', 'ALT 48', 'ALT after 24 w', 'RNA EOT', 'Baseline histological Grading','outlier']
                                 , errors='ignore')

In [ ]:
# defining training and target features

X = df_clean_iso.drop(columns=['Baselinehistological staging'], axis=1)
y = df_clean_iso['Baselinehistological staging']

# Train - test split\

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Logistic regression

In [ ]:
# defining logistic regression pipeline
log_pipeline = Pipeline([
    ('transformer', QuantileTransformer(output_distribution = 'normal', random_state=42)),
    ('classifier', LogisticRegression(penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced",
        random_state=42 ))])

# training pipeline
log_pipeline.fit(X_train, y_train)

# testing pipeline
log_preds = log_pipeline.predict(X_test)

# evaluating predictions
log_results = classification_report(y_test, log_preds)
print(log_results)

The model performs poorly, with only 21% accuracy and low F1-scores across all fibrosis stages, indicating it is barely better than random guessing for a four-class classification problem.

# Calibration plot - Logistic Regression

In [ ]:
# getting predicted probabilities from logistic regression
log_probs = log_pipeline.predict_proba(X_test)

# getting the class labels
classes = log_pipeline.classes_

# binarising y_test for one-vs-rest calibration plotting
y_test_bin = label_binarize(y_test, classes=classes)

# plotting calibration curve
plt.figure(figsize=(8, 6))

# loop through each fibrosis stage
for i, class_label in enumerate(classes):

    # calculate calibration curve for each class
    prob_true, prob_pred = calibration_curve(
        y_test_bin[:, i],
        log_probs[:, i],
        n_bins=10,
        strategy='uniform'
    )

    # plot reliability curve
    plt.plot(prob_pred, prob_true, marker='o', label=f'Stage {class_label}')

# perfect calibration reference line
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='Perfect Calibration')

# formatting plot
plt.title('Calibration Plot - Logistic Regression')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend()
plt.grid(True)
plt.show()

# Error vs feature distribution - logistic regression

Data preparation

In [ ]:
# creating a copy of the test data
log_error_df = X_test.copy()

# adding actual and predicted fibrosis stages
log_error_df['Actual_Stage'] = y_test.values
log_error_df['Predicted_Stage'] = log_preds

# checking if the prediction was correct or incorrect
log_error_df['Prediction_Result'] = np.where(
    log_error_df['Actual_Stage'] == log_error_df['Predicted_Stage'],
    'Correct',
    'Incorrect'
)

# checking the dataframe
log_error_df.head()

Error plots - logistic regression

In [ ]:
# choosing selected features to inspect
features_to_plot = [col for col in X_train.columns if col in numerical_columns]

# setting number of plots per row
plots_per_row = 3

# calculating number of rows needed
n_rows = math.ceil(len(features_to_plot) / plots_per_row)

# creating subplots
fig, axes = plt.subplots(n_rows, plots_per_row, figsize=(18, n_rows * 4))

# flattening axes so they can be looped through easily
axes = axes.flatten()

# loop through each feature
for i, feature in enumerate(features_to_plot):

    # plotting correct vs incorrect prediction distributions
    sns.kdeplot(
        data=log_error_df,
        x=feature,
        hue='Prediction_Result',
        fill=True,
        common_norm=False,
        alpha=0.4,
        ax=axes[i]
    )

    # formatting each plot
    axes[i].set_title(f'Error vs {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Density')
    axes[i].grid(True)

# removing empty subplot spaces
for j in range(len(features_to_plot), len(axes)):
    fig.delaxes(axes[j])

# formatting full figure
plt.suptitle('Baseline Model Error vs Feature Distribution', fontsize=16)
plt.tight_layout()
plt.show()

# Random forest

In [ ]:
# defining random forest model
rf_model = RandomForestClassifier(
    n_estimators=200,        # number of trees
    max_depth=None,          # let trees grow fully
    min_samples_split=2,     # minimum samples to split
    min_samples_leaf=1,      # minimum samples per leaf
    max_features='sqrt',     # features considered per split
    class_weight='balanced',
    random_state=42,
    n_jobs=-1)

# training model
rf_model.fit(X_train, y_train)

# predictions
rf_preds = rf_model.predict(X_test)

# evaluating predictions
rf_results = classification_report(y_test, rf_preds)
print(rf_results)

This model also performs poorly, with 24% accuracy and macro F1 of 0.23, showing weak class separation and no reliable ability to predict fibrosis stage.

Calibration plot - Random Forest

In [ ]:
# getting predicted probabilities from random forest
rf_probs = rf_model.predict_proba(X_test)

# getting the class labels
classes = rf_model.classes_

# binarising y_test for one-vs-rest calibration plotting
y_test_bin = label_binarize(y_test, classes=classes)

# plotting calibration curve
plt.figure(figsize=(8, 6))

# loop through each fibrosis stage
for i, class_label in enumerate(classes):

    # calculate calibration curve for each class
    prob_true, prob_pred = calibration_curve(
        y_test_bin[:, i],
        rf_probs[:, i],
        n_bins=10,
        strategy='uniform'
    )

    # plot reliability curve
    plt.plot(prob_pred, prob_true, marker='o', label=f'Stage {class_label}')

# perfect calibration reference line
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='Perfect Calibration')

# formatting plot
plt.title('Calibration Plot - Random Forest')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend()
plt.grid(True)
plt.show()

# Error vs feature plot - random forest

Data preparation

In [ ]:
# creating a copy of the test data
rf_error_df = X_test.copy()

# adding actual and predicted fibrosis stages
rf_error_df['Actual_Stage'] = y_test.values
rf_error_df['Predicted_Stage'] = rf_preds

# checking if the prediction was correct or incorrect
rf_error_df['Prediction_Result'] = np.where(
    rf_error_df['Actual_Stage'] == rf_error_df['Predicted_Stage'],
    'Correct',
    'Incorrect'
)

# checking the dataframe
rf_error_df.head()

Error plots - random forest

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math

# choosing selected features to inspect
features_to_plot = [col for col in X_train.columns if col in numerical_columns]

# setting number of plots per row
plots_per_row = 3

# calculating number of rows needed
n_rows = math.ceil(len(features_to_plot) / plots_per_row)

# creating subplots
fig, axes = plt.subplots(n_rows, plots_per_row, figsize=(18, n_rows * 4))

# flattening axes so they can be looped through easily
axes = axes.flatten()

# loop through each feature
for i, feature in enumerate(features_to_plot):

    # plotting correct vs incorrect prediction distributions
    sns.kdeplot(
        data=rf_error_df,
        x=feature,
        hue='Prediction_Result',
        fill=True,
        common_norm=False,
        alpha=0.4,
        ax=axes[i]
    )

    # formatting each plot
    axes[i].set_title(f'Error vs {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Density')
    axes[i].grid(True)

# removing empty subplot spaces
for j in range(len(features_to_plot), len(axes)):
    fig.delaxes(axes[j])

# formatting full figure
plt.suptitle('Random Forest Error vs Feature Distribution', fontsize=16)
plt.tight_layout()
plt.show()

# Modelling using descretized data

Creating descretization bins

In [ ]:
# copy the cleaned dataframe to avoid altering original data
df_discrete = df_clean_iso.copy()

# discretize age into clinical 5-year intervals
age_bins = [0, 32, 37, 42, 47, 52, 57, 62, 100]
age_labels = ['<=32', '33-37', '38-41', '42-46', '47-51', '52-56', '57-61', '>=62']
df_discrete['Age_Bin'] = pd.cut(df_discrete['Age '], bins=age_bins, labels=age_labels, right=False)

# discretize bmi categories
bmi_bins = [0, 18.5, 25, 30, 35, 40, 100]
bmi_labels = ['Underweight', 'Normal', 'Overweight', 'Obese 1', 'Obese 2', 'Extreme']
df_discrete['BMI_Bin'] = pd.cut(df_discrete['BMI'], bins=bmi_bins, labels=bmi_labels, right=False)

# discretize wbc and rbc counts
df_discrete['WBC_Bin'] = pd.cut(df_discrete['WBC'], bins=[0, 4000, 11000, 100000], labels=['Low', 'Normal', 'High'], right=False)
df_discrete['RBC_Bin'] = pd.cut(df_discrete['RBC'], bins=[0, 3000000, 5000000, 10000000], labels=['Low', 'Normal', 'High'], right=False)

# discretize hgb based on gender-specific clinical thresholds (1 = male, 2 = female)
def discretize_hgb(row):
    if row['Gender'] == 1:
        if row['HGB'] < 14: return 'Low'
        elif 14 <= row['HGB'] < 17.5: return 'Normal'
        else: return 'High'
    else:
        if row['HGB'] < 12.3: return 'Low'
        elif 12.3 <= row['HGB'] < 15.3: return 'Normal'
        else: return 'High'

df_discrete['HGB_Bin'] = df_discrete.apply(discretize_hgb, axis=1)

# discretize liver enzymes (ast/alt) into low, normal, high
liver_enzymes = ['AST 1', 'ALT 1', 'ALT4', 'ALT 12', 'ALT 24']
for col in liver_enzymes:
    if col in df_discrete.columns:
        df_discrete[col + '_Bin'] = pd.cut(df_discrete[col], bins=[0, 20, 40, 500], labels=['Low', 'Normal', 'High'], right=False)

# discretize rna viral load into tertiles (low, medium, high)
rna_cols = ['RNA Base', 'RNA 4', 'RNA 12']
for col in rna_cols:
    if col in df_discrete.columns:
        df_discrete[col + '_Bin'] = pd.qcut(df_discrete[col], q=3, labels=['Low', 'Medium', 'High'], duplicates='drop')

# isolate engineered categorical columns alongside original symptoms and target
cols_to_keep = [col for col in df_discrete.columns if '_Bin' in col] + \
               ['Gender', 'Fever', 'Nausea/Vomting', 'Headache ', 'Diarrhea ',
                'Fatigue & generalized bone ache ', 'Jaundice ', 'Epigastric pain ',
                'Baselinehistological staging']

df_model = df_discrete[cols_to_keep].copy()

# drop any residual missing values created by binning
df_model = df_model.dropna()

Encoding and data preparation

In [ ]:
# initialize label encoder
le = LabelEncoder()

# encode all categorical string bins into numeric values
for col in df_model.columns:
    if df_model[col].dtype == 'category' or df_model[col].dtype == 'object':
        df_model[col] = le.fit_transform(df_model[col].astype(str))

# define features (X) and target (y)
X = df_model.drop(columns=['Baselinehistological staging'])
y = df_model['Baselinehistological staging']

# split data into training and test sets (stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Model training and testing

In [ ]:
# train a decision tree
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)

# print decision tree evaluation metrics
print("--- Decision Tree (Rule-Mining Approximation) ---")
print("Accuracy:", accuracy_score(y_test, dt_preds))
print(classification_report(y_test, dt_preds))

# train a categorical random forest (ensemble of rules to prevent overfitting)
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

# print random forest evaluation metrics
print("\n--- Categorical Random Forest ---")
print("Accuracy:", accuracy_score(y_test, rf_preds))
print(classification_report(y_test, rf_preds))

Calibration plot - decision tree

In [ ]:
# getting predicted probabilities from decision tree
dt_probs = dt_model.predict_proba(X_test)

# getting the class labels
classes = dt_model.classes_

# binarising y_test for one-vs-rest calibration plotting
y_test_bin = label_binarize(y_test, classes=classes)

# plotting calibration curve
plt.figure(figsize=(8, 6))

# loop through each fibrosis stage
for i, class_label in enumerate(classes):

    # calculate calibration curve for each class
    prob_true, prob_pred = calibration_curve(
        y_test_bin[:, i],
        dt_probs[:, i],
        n_bins=10,
        strategy='uniform'
    )

    # plot reliability curve
    plt.plot(prob_pred, prob_true, marker='o', label=f'Stage {class_label}')

# perfect calibration reference line
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='Perfect Calibration')

# formatting plot
plt.title('Calibration Plot - Decision Tree')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend()
plt.grid(True)
plt.show()

Calibration plot - Random Forest

In [ ]:
# getting predicted probabilities from categorical random forest
rf_cat_probs = rf_model.predict_proba(X_test)

# getting the class labels
classes = rf_model.classes_

# binarising y_test for one-vs-rest calibration plotting
y_test_bin = label_binarize(y_test, classes=classes)

# plotting calibration curve
plt.figure(figsize=(8, 6))

# loop through each fibrosis stage
for i, class_label in enumerate(classes):

    # calculate calibration curve for each class
    prob_true, prob_pred = calibration_curve(
        y_test_bin[:, i],
        rf_cat_probs[:, i],
        n_bins=10,
        strategy='uniform'
    )

    # plot reliability curve
    plt.plot(prob_pred, prob_true, marker='o', label=f'Stage {class_label}')

# perfect calibration reference line
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='Perfect Calibration')

# formatting plot
plt.title('Calibration Plot - Categorical Random Forest')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend()
plt.grid(True)
plt.show()

# **t-SNE**

Data transformation

In [ ]:
# transforming features before t-SNE
qt = QuantileTransformer(output_distribution='normal', random_state=42)

X_tsne_ready = qt.fit_transform(X)

# checking transformed shape
print(X_tsne_ready.shape)

t-SNE model

In [ ]:
# defining t-SNE model
tsne_model = TSNE(
    n_components=2,      # reduce into 2 dimensions
    perplexity=30,       # common starting point
    learning_rate='auto',
    init='pca',
    random_state=42
)

# fitting and transforming the data
X_tsne = tsne_model.fit_transform(X_tsne_ready)

# putting t-SNE result into a dataframe
tsne_df = pd.DataFrame(X_tsne, columns=['tSNE_1', 'tSNE_2'])

# adding fibrosis stage labels
tsne_df['Baselinehistological staging'] = y.values

# checking first few rows
tsne_df.head()

t-SNA plot

In [ ]:
# plotting t-SNE visualisation
plt.figure(figsize=(9, 7))

# loop through each fibrosis stage
for stage in sorted(tsne_df['Baselinehistological staging'].unique()):

    # filtering points for each stage
    subset = tsne_df[tsne_df['Baselinehistological staging'] == stage]

    # plotting the points
    plt.scatter(
        subset['tSNE_1'],
        subset['tSNE_2'],
        label=f'Stage {stage}',
        alpha=0.7
    )

# formatting plot
plt.title('t-SNE Visualisation of Fibrosis Stages')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend()
plt.grid(True)
plt.show()

The t-SNE plot shows heavy overlap between all four fibrosis stages, suggesting that the available clinical features do not naturally separate patients into distinct stage-specific clusters.